# Section 1 Assignment - Transformers

# Task
You work for a large company and the company receives 8 letters regarding one of your products. These letters range from letters of complaint to letters of praise.


Using Hugging Face Transformers,
1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the positive responses and negative responses.
1. Summarize the sentiments towards the product and
1. Provide a summary of the letters that reflect the most extreme sentiments.
1. Determine and report in each case on:
    - What the customer wants to happen by automatically questioning their respective letters.
1. Knowing how each customer feels,
    - Automatically generate replies to the two most extreme examples, including their name in the greeting in the response.

1. In your code to generate the report for your manager and the letters of reply,
    - indicate your design choices and
    - use any pre-processing trick that you anticipate will improve your automatic generation.


## Tools for the task
* Some great working examples can be found in the file **01_introduction.ipynb** from the [Natural Language Processing with Transformers GitHub Repository](https://github.com/nlp-with-transformers/notebooks).
* You will need to generate/provide the letters of complaint. You can generate the letters from the customers manually or automatically. You can use a Hugging Face text generation pipeline to do this or an online generator like [You.com](https://you.com/search?q=how+to+write+well&&tbm=youwrite&cfr=write&).
* You can use the library such as [python-docx](https://python-docx.readthedocs.io/en/latest/index.html#) to generate the report and the response letters.
* There is a variety of models on Hugging Face for the different tasks, e.g, [question-answering](https://huggingface.co/models?pipeline_tag=question-answering&sort=downloads) that allow you to try out different models for your task. It is strongly recommended that you review these and look at the model cards in each case. Also, you can test your text in the Hosted inference API on the right hand side of each model.

## Environment setup

In [5]:
#!pip install -q transformers torch sentencepiece

In [15]:
from transformers import pipeline
import pandas as pd
import textwrap

### Build Summarisation Model

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load BART for summarization
summary_model_name = "facebook/bart-large-cnn"
summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(summary_model_name)


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [10]:
def summarise_text(model, tokenizer, text):

    # Generate summary
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=150,      # Use max_new_tokens, not max_length
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print("Summary:", summary)

    return summary

In [13]:
summary_example = summarise_text(summary_model, summary_tokenizer, text)

print (summary_example)

Summary: Bumblebee ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead. As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange ofMegatron for the Optimus Prime figure.
Bumblebee ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead. As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange ofMegatron for the Optimus Prime figure.


In [16]:
summary_format = textwrap.fill(summary_example, width=80)
print(summary_format)

Bumblebee ordered an Optimus Prime action figure from your online store in
Germany. Unfortunately, when I opened the package, I discovered to my horror
that I had been sent an action figure of Megatron instead. As a lifelong enemy
of the Decepticons, I hope you can understand my dilemma. To resolve the issue,
I demand an exchange ofMegatron for the Optimus Prime figure.


## Build Sentiment Model

In [17]:
from transformers import pipeline

classifier = pipeline("text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [18]:
def determine_sentiment(classifier, text):
    outputs = classifier(text)

    for output in outputs:
        label = output["label"]
        score = output["score"]

    return outputs

In [19]:
print(determine_sentiment(classifier, text))

[{'label': 'NEGATIVE', 'score': 0.9015460014343262}]


# Workings


In [7]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

In [8]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

outputs = summarizer(text, max_length=130, min_length=30, do_sample=False)
#outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
### Letter of complaint
generator = pipeline("text-generation", model="gpt2")

letter_gen_prompt= """
Generate 8 customer letters about an ipad.
Letters 1-4 should complain about the widget.  Vary the level of complaint within these letters ranging from a mild complaint to a serious complaint.
Letters 5-8 should praise the widget.  Vary the level of satisfaction witin these letters ranging from a mild satisfaction to complete satisfaction.

Include a customer name in each letter.

Start each letter with

Dear Customer service,
"""

result = generator(letter_gen_prompt, temperature=0.8)


letters = result[0]["generated_text"].split("Letter")
df = pd.DataFrame(result)
df


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,generated_text
0,\nGenerate 8 customer letters about an ipad. \...


In [ ]:
for index, letter in enumerate(letters):
    print(f'Letter {index+1}:')
    print(f'{letter}')

Letter 1:

Generate 8 customer letters about an ipad. 

Letter 2:
s 1-4 should complain about the widget.  Vary the level of complaint within these letters ranging from a mild complaint to a serious complaint.

Letter 3:
s 5-8 should praise the widget.  Vary the level of satisfaction witin these letters ranging from a mild satisfaction to complete satisfaction.

Include a customer name in each letter.

Start each letter with 

Dear Customer service,

I am wondering if you can send an email address to the customer service team and advise them on if we are happy with the widget. My recommendation would be to send an email to the customer service team to confirm that it is not for my personal use. I would also like to include the name of the widget on the widget.

We would also like to include the name of the widget on the widget.

As an optional bonus, we would like to include the name of the widget on their website.

Since this is only a suggestion, I don't want to include this informat